In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. Haversine Distance Function
# ---------------------------------------------------------
def haversine_distance(lat1, lon1, lat2, lon2):
    r = 6371  # Earth's radius in kilometers
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)

    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

# ---------------------------------------------------------
# 2. Data Preprocessing & Feature Engineering
# ---------------------------------------------------------
# Load dataset
df = pd.read_csv('uber.csv')  #give appropriate path

# Drop missing rows
df.dropna(inplace=True)

# Calculate distance using Haversine
df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

# Extract datetime features
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

# Outlier Filtering (Remove non-sensical values)
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['distance_km'] > 0) & (df['distance_km'] < 100)]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]

# ---------------------------------------------------------
# 3. Model Training & Evaluation
# ---------------------------------------------------------
X = df[['distance_km', 'passenger_count', 'hour', 'day_of_week']]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Print Evaluation Metrics
print("--- Linear Regression Performance ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_lr):.4f}")

print("\n--- Random Forest Regression Performance ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_rf):.4f}")


--- Linear Regression Performance ---
RMSE: 5.7260
R2 Score: 0.6654

--- Random Forest Regression Performance ---
RMSE: 5.3367
R2 Score: 0.7093
